# Jiaqi's papers

Collaging all of Jiaqi's papers and collating them into a pdf book.

In [1]:
import os
import requests
import pandas as pd
import time
import re
import sqlite3

## Fetch metadata from ORCID

Use a few ORCID APIs to get the details of the papers.  Thanks copilot!
It creates a csv file [papers.csv](papers.csv) and an sqlite3 database [papers.db](papers.db)

In [3]:

# --------------------------
# Configuration
# --------------------------

ORCID_ID = "0000-0001-6491-3851"
BASE = f"https://pub.orcid.org/v3.0/{ORCID_ID}"
HEADERS = {"Accept": "application/json"}
SLEEP_BETWEEN_CALLS = 0.2  # be polite to ORCID’s API
USE_CROSSREF = True        # set False to disable Crossref fallback
CROSSREF_CONTACT_EMAIL = "your_email@leeds.ac.uk"  # required in UA per Crossref etiquette

DOI_RE = re.compile(r"(10\.\d{4,9}/\S+)", re.IGNORECASE)


# --------------------------
# Helpers
# --------------------------

def normalize_doi(doi):
    """Strip DOI URLs -> bare DOI, trim whitespace/punctuation."""
    if not doi:
        return None
    doi = doi.strip()
    doi = re.sub(r"^https?://(dx\.)?doi\.org/", "", doi, flags=re.IGNORECASE)
    return doi.strip(" ,.;")


def fetch_full_work(put_code):
    """Fetch full ORCID work record."""
    url = f"{BASE}/work/{put_code}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()


def extract_ext_ids(full):
    """
    Normalise external-id node in full ORCID metadata.
    external-id may be: None, dict, list of dicts.
    """
    ext_ids = full.get("external-ids", {}).get("external-id")
    if ext_ids is None:
        return []
    if isinstance(ext_ids, dict):
        return [ext_ids]
    if isinstance(ext_ids, list):
        return [e for e in ext_ids if isinstance(e, dict)]
    return []


def extract_id(full, id_type):
    """Get an identifier by type (e.g., 'issn', 'eissn', 'isbn', 'doi')."""
    id_type = (id_type or "").lower()
    for eid in extract_ext_ids(full):
        typ = (eid.get("external-id-type") or "").lower()
        if typ == id_type:
            return (eid.get("external-id-value") or "").strip()
    return None


def extract_doi_from_ext_ids(full):
    for eid in extract_ext_ids(full):
        if (eid.get("external-id-type") or "").lower() == "doi":
            return normalize_doi(eid.get("external-id-value"))
    return None


def extract_doi_from_citation(full):
    """Extract DOI from citation free-text block. Handles null citations."""
    cit = full.get("citation")
    if not isinstance(cit, dict):
        return None
    cit_val = cit.get("citation-value")
    if not isinstance(cit_val, str):
        return None
    m = DOI_RE.search(cit_val)
    if m:
        return normalize_doi(m.group(1))
    return None


def extract_doi_from_url(full):
    """Extract DOI if URL is something like https://doi.org/..."""
    url_obj = full.get("url")
    u = url_obj.get("value") if isinstance(url_obj, dict) else url_obj
    if isinstance(u, str) and "doi.org/" in u.lower():
        return normalize_doi(u)
    return None


def extract_pub_date_parts(full):
    """Extract year, month, day safely."""
    year = month = day = None
    pd = full.get("publication-date")
    if isinstance(pd, dict):
        y = pd.get("year"); m = pd.get("month"); d = pd.get("day")
        if isinstance(y, dict): year = y.get("value")
        if isinstance(m, dict): month = m.get("value")
        if isinstance(d, dict): day = d.get("value")
    return year, month, day


def get_doi(full):
    """Try all DOI sources in logical order."""
    doi = extract_doi_from_ext_ids(full)
    if doi: return doi
    doi = extract_doi_from_citation(full)
    if doi: return doi
    doi = extract_doi_from_url(full)
    if doi: return doi
    return None


def extract_citation_fields(full):
    """
    Return (citation_type, citation_value). ORCID citation is optional and may hold
    structured strings like APA, BIBTEX, etc.
    """
    cit = full.get("citation")
    if isinstance(cit, dict):
        ctype = cit.get("citation-type")
        cval = cit.get("citation-value")
        if isinstance(cval, str):
            return ctype, cval
    return None, None


def extract_authors(full):
    """
    Extract ordered authors from ORCID contributors.
    Prefer credit-name, else given + family. Keep order by contributor-sequence if available.
    """
    out = []
    contribs = (full.get("contributors") or {}).get("contributor") or []
    if not isinstance(contribs, list):
        contribs = []
    def seq_val(c):
        seq = ((c.get("contributor-attributes") or {}).get("contributor-sequence") or "")
        # Try to convert typical numeric sequences; otherwise keep large value to preserve input order
        try:
            return int(seq)
        except Exception:
            return 1_000_000  # unknown -> end
    # sort by sequence if present
    contribs_sorted = sorted(contribs, key=seq_val)
    for c in contribs_sorted:
        # role filter (keep all if role missing)
        role = ((c.get("contributor-attributes") or {}).get("contributor-role") or "").lower()
        if role and role not in {"author", "co-investigator", "editor"}:
            # keep authors and near-author roles; you can tighten this if you want strictly 'author'
            pass
        # pick a name
        credit = (c.get("credit-name") or {}).get("value")
        if credit:
            out.append(credit.strip())
            continue
        name = c.get("contributor-name") or {}
        given = (name.get("given-names") or {}).get("value")
        family = (name.get("family-name") or {}).get("value")
        if given or family:
            out.append(" ".join([p for p in [given, family] if p]).strip())
    # Deduplicate while preserving order
    seen = set()
    ordered = []
    for a in out:
        if a and a not in seen:
            ordered.append(a)
            seen.add(a)
    return ordered


def safe_url(full):
    url_obj = full.get("url")
    u = url_obj.get("value") if isinstance(url_obj, dict) else url_obj
    return u if isinstance(u, str) else None


def orcid_abstract(full):
    """ORCID 'short-description' may contain an abstract-like text."""
    sd = full.get("short-description")
    return sd if isinstance(sd, str) and sd.strip() else None


# ---- Crossref fallback (optional) ----

def crossref_fetch(doi):
    """Fetch Crossref metadata for DOI to fill in abstract/volume/issue/pages/container & authors."""
    if not USE_CROSSREF or not doi:
        return {}
    url = f"https://api.crossref.org/works/{doi}"
    headers = {"User-Agent": f"ORCID-Book-Builder (mailto:{CROSSREF_CONTACT_EMAIL})"}
    try:
        r = requests.get(url, headers=headers, timeout=30)
        if r.status_code != 200:
            return {}
        return r.json().get("message", {}) or {}
    except Exception:
        return {}


def strip_tags(text):
    """Remove JATS/HTML tags from Crossref abstracts."""
    if not isinstance(text, str):
        return None
    # Crossref often wraps abstracts in <jats:p>...</jats:p>
    cleaned = re.sub(r"<[^>]+>", " ", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or None


def merge_with_crossref_if_needed(orcid_meta, cr):
    """
    Fill gaps using Crossref message dict.
    Fields considered: abstract, container-title, volume, issue, page, authors, year/month/day.
    """
    updated = dict(orcid_meta)  # copy

    # abstract
    if not updated.get("abstract"):
        updated["abstract"] = strip_tags(cr.get("abstract"))

    # container title (journal)
    if not updated.get("journal"):
        ct = cr.get("container-title") or []
        if isinstance(ct, list) and ct:
            updated["journal"] = ct[0]

    # volume / issue / pages
    updated["volume"] = updated.get("volume") or cr.get("volume")
    updated["issue"]  = updated.get("issue")  or cr.get("issue")
    updated["pages"]  = updated.get("pages")  or cr.get("page")

    # authors (list of dicts with given/family)
    if not updated.get("authors"):
        auths = []
        for a in cr.get("author", []) or []:
            given = a.get("given"); family = a.get("family")
            name = " ".join([p for p in [given, family] if p]).strip()
            if name:
                auths.append(name)
        if auths:
            updated["authors"] = auths

    # date parts
    if not updated.get("year"):
        # Crossref 'issued' -> date-parts [[Y, M, D?]]
        dp = ((cr.get("issued") or {}).get("date-parts") or [[]])
        if dp and dp[0]:
            updated["year"]  = str(dp[0][0]) if len(dp[0]) >= 1 else None
            updated["month"] = str(dp[0][1]) if len(dp[0]) >= 2 else None
            updated["day"]   = str(dp[0][2]) if len(dp[0]) >= 3 else None

    return updated


# --------------------------
# Main execution
# --------------------------

if os.path.exists("papers.csv"):# and os.path.exists("papers.db"):
    print("papers.csv|db already exists — skipping ORCID download.")
else:
    print("papers.csv not found — fetching from ORCID...")

    print("Fetching list of works…")
    summary = requests.get(f"{BASE}/works", headers=HEADERS, timeout=30).json()

    # Collect put-codes
    put_codes = []
    for group in summary.get("group", []):
        for w in group.get("work-summary", []):
            put_codes.append(w["put-code"])

    print(f"Found {len(put_codes)} works. Fetching full records…\n")

    records = []

    for pc in put_codes:
        full = fetch_full_work(pc)

        # Basic ORCID fields
        title   = (full.get("title") or {}).get("title", {}) or {}
        title   = title.get("value")
        subtitle = None  # available as 'subtitle' but seldom used
        year, month, day = extract_pub_date_parts(full)
        journal = (full.get("journal-title") or {}).get("value")
        doi     = get_doi(full)
        url     = safe_url(full)

        # identifiers
        issn  = extract_id(full, "issn")
        eissn = extract_id(full, "eissn") or extract_id(full, "e-issn")
        isbn  = extract_id(full, "isbn")

        # citation
        citation_type, citation_value = extract_citation_fields(full)

        # authors
        authors = extract_authors(full)

        # abstract
        abstract = orcid_abstract(full)

        # placeholders for volume/issue/pages (not really in ORCID; may come from Crossref)
        volume = issue = pages = None

        # assemble initial metadata
        meta = {
            "put_code": pc,
            "type": full.get("type"),
            "title": title,
            "subtitle": subtitle,
            "year": year,
            "month": month,
            "day": day,
            "journal": journal,
            "volume": volume,
            "issue": issue,
            "pages": pages,
            "doi": doi,
            "url": url,
            "issn": issn,
            "eissn": eissn,
            "isbn": isbn,
            "authors": "; ".join(authors) if authors else None,
            "citation_type": citation_type,
            "citation_value": citation_value,
            "abstract": abstract,
            "abstract_source": "orcid" if abstract else None,
        }

        # Crossref fallback to fill gaps
        if USE_CROSSREF and doi:
            cr = crossref_fetch(doi)
            if cr:
                before = dict(meta)
                meta = merge_with_crossref_if_needed(meta, cr)
                if meta.get("abstract") and not before.get("abstract"):
                    meta["abstract_source"] = "crossref"

                # Fill volume/issue/pages if found
                # (already handled in merge; nothing else needed)

        records.append(meta)

        short_title = title[:60] if title else "No title"
        print(f"Fetched {pc}: {short_title}  --> DOI: {doi}")
        time.sleep(SLEEP_BETWEEN_CALLS)

    df = pd.DataFrame(records)
    df.to_csv("papers.csv", index=False)

    ## Persist to SQLite for convenience
    #conn = sqlite3.connect("papers.db")
    #try:
    #    df.to_sql("papers", conn, if_exists="replace", index=False)
    #finally:
    #    conn.close()

    print("\nDone. Saved papers.csv and created papers.db")


papers.csv|db already exists — skipping ORCID download.


## Download papers (that are available publicly)

In [39]:
import json

df = pd.read_csv("papers.csv")
os.makedirs("pdfs", exist_ok=True)

for _, row in df.iterrows():
    doi = row["doi"]
    print(f"DOI: {doi}")
    if pd.isna(doi):
        continue

    
    api = f"https://api.unpaywall.org/v2/{doi}?email=n.s.malleson@leeds.ac.uk"
    resp = requests.get(api, timeout=20)

    try:
        r = resp.json()
    except json.JSONDecodeError:
        print(f"\tUnpaywall returned invalid JSON for {doi}. Status code {resp.status_code}. Skipping.")
        #print("\t",resp.text[:200])
        continue

    oa = r.get("best_oa_location") or {}
    pdf_url = oa.get("url_for_pdf")


    if pdf_url:
        pdf_data = requests.get(pdf_url).content
        filename = f"pdfs/{doi.replace('/', '_')}.pdf"
        with open(filename, "wb") as f:
            f.write(pdf_data)
        print("\tDownloaded", filename)
    else:
        print("\tNot available")


DOI: 10.1061/jtepbs.teeng-9272
	Not available
DOI: 10.1371/journal.pstr.0000162
	Downloaded pdfs/10.1371_journal.pstr.0000162.pdf
DOI: 10.25937/8bf3-h968
	Unpaywall returned invalid JSON for 10.25937/8bf3-h968. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try agai
DOI: 10.17632/9v4byyvgxh.1
	Unpaywall returned invalid JSON for 10.17632/9v4byyvgxh.1. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try agai
DOI: 10.4230/LIPIcs.GIScience.2023.78
	Unpaywall returned invalid JSON for 10.4230/LIPIcs.GIScience.2023.78. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you e

## Create latex

There is a template file called [papers_template.tex](papers_template.tex). Now populate it with the papers. 
This is slightly more complicated than I thought it would be at first because copilot was quite careful with formatting the references.

In [28]:

import os
import re
import pandas as pd

df = pd.read_csv("papers.csv")

def latex_escape(text: str) -> str:
    """Escape LaTeX special chars in plain text."""
    if not isinstance(text, str):
        return ""
    # Escape backslash first
    reps = [
        (r'\\', r'\\textbackslash{}'),
        (r'&', r'\&'),
        (r'%', r'\%'),
        (r'\$', r'\$'),
        (r'#', r'\#'),
        (r'_', r'\_'),
        (r'\{', r'\{'),
        (r'\}', r'\}'),
        (r'~', r'\textasciitilde{}'),
        (r'\^', r'\textasciicircum{}'),
    ]
    out = text
    for a, b in reps:
        out = re.sub(a, b, out)
    return out

def normalize_space(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s).strip()

def format_authors(authors_cell: str, max_authors: int = 20) -> str:
    """
    Input is a semicolon-separated string "A. Author; B. Writer; C. Researcher".
    Returns the same, optionally truncating with 'et al.' if excessive.
    """
    if not isinstance(authors_cell, str) or not authors_cell.strip():
        return ""
    parts = [normalize_space(a) for a in authors_cell.split(";") if normalize_space(a)]
    if not parts:
        return ""
    if len(parts) > max_authors:
        parts = parts[:max_authors]
        return "; ".join(parts) + "; et al."
    return "; ".join(parts)

def apa_like_citation(row) -> str:
    """
    Build a readable, LaTeX-safe citation like:
      Authors (Year). Title. \textit{Journal}, Volume(Issue), Pages. https://doi.org/DOI
    Missing pieces are omitted cleanly.
    """
    authors = format_authors(row.get("authors", ""))
    year    = str(row.get("year") or "").strip()
    title   = normalize_space(row.get("title") or "")
    journal = normalize_space(row.get("journal") or "")
    volume  = str(row.get("volume") or "").strip()
    issue   = str(row.get("issue") or "").strip()
    pages   = normalize_space(str(row.get("pages") or "").strip())
    doi     = normalize_space(row.get("doi") or "")

    parts = []

    if authors:
        parts.append(latex_escape(authors) + ("" if authors.endswith(".") else "."))

    if year:
        parts.append(
                f"({latex_escape(str(int(float(v))))})" 
                if (v := str(row.get("year") or "").strip()) else ""
        )
        

    if title:
        # Add sentence period if not present
        t = latex_escape(title)
        parts.append(t + ("" if t.endswith(".") else "."))

    # Journal piece (italic journal; add volume/issue/pages)
    vip_bits = []
    if journal:
        vip_bits.append(r"\textit{" + latex_escape(journal) + "}")
    if volume and issue:
        vip_bits.append(latex_escape(f"{volume}({issue})"))
    elif volume:
        vip_bits.append(latex_escape(volume))
    if pages:
        vip_bits.append(latex_escape(pages))

    if vip_bits:
        parts.append(", ".join(vip_bits) + ".")

    if doi:
        parts.append(latex_escape(f"https://doi.org/{doi}"))

    return " ".join(parts).strip()

def abstract_text(row) -> str:
    abs_txt = row.get("abstract")
    if isinstance(abs_txt, str) and abs_txt.strip():
        return latex_escape(normalize_space(abs_txt))
    return ""

with open("chapters.tex", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        doi = row.get("doi")

        # Skip rows without a DOI (your pipeline names PDFs by DOI)
        if pd.isna(doi) or not isinstance(doi, str) or not doi.strip():
            continue

        title = row.get("title") or ""
        f.write(f"\\chapter{{{latex_escape(title)}}}\n\n")

        # --- Citation (text-ready) ---
        citation_line = apa_like_citation(row)
        if citation_line:
            f.write("\\noindent\\textbf{Citation:} " + citation_line + "\n\n")

        # --- Abstract (if present) ---
        abs_txt = abstract_text(row)
        if abs_txt:
            f.write("\\noindent\\textbf{Abstract}\\par\n")
            f.write("\\begin{quote}\\small\n")
            f.write(abs_txt + "\n")
            f.write("\\end{quote}\n\n")

        # --- PDF Include or 'No PDF' ---
        pdf_file = f"pdfs/{doi.replace('/', '_')}.pdf"
        if os.path.exists(pdf_file):
            f.write("\\vspace{0.5em}\n")
            f.write(f"\\includepdf[pages=-]{{{pdf_file}}}\n\n")
        else:
            f.write("\\vspace{0.5em}\n")
            #f.write("\\noindent\\textit{No PDF}\n\n")
